# Price Coverage Diagnostics

Use this notebook after the audit notebook to diagnose why selected symbols start late or have missing price history.

This notebook pulls evidence from silver facts and bronze key metadata.

In [1]:
import os
import re
import sys
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(PROJECT_ROOT / '.env')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from ingestion.utils import r2_client

BUCKET = os.environ['R2_BUCKET_NAME']
ACCT = os.environ['R2_ACCOUNT_ID']
AK = os.environ['R2_ACCESS_KEY_ID']
SK = os.environ['R2_SECRET_ACCESS_KEY']

def open_con() -> duckdb.DuckDBPyConnection:
    c = duckdb.connect()
    c.execute('INSTALL httpfs; LOAD httpfs;')
    c.execute(f"SET s3_endpoint = '{ACCT}.r2.cloudflarestorage.com';")
    c.execute(f"SET s3_access_key_id = '{AK}';")
    c.execute(f"SET s3_secret_access_key = '{SK}';")
    c.execute("SET s3_region = 'auto';")
    c.execute("SET s3_url_style = 'path';")
    return c

def silver(path: str) -> str:
    return f"read_parquet('s3://{BUCKET}/silver/{path}', union_by_name=true)"

con = open_con()
print('Connected to R2 via DuckDB and r2_client')

Connected to R2 via DuckDB and r2_client


In [2]:
# Parameters
FLOOR_DATE = '1999-01-01'
SYMBOLS = ['CMG', 'SBUX', 'MCD']  # replace with symbols flagged in audit

symbols_sql = '(' + ', '.join([f"'{s}'" for s in SYMBOLS]) + ')'
print('Diagnostics symbols:', SYMBOLS)

Diagnostics symbols: ['CMG', 'SBUX', 'MCD']


In [3]:
# Silver coverage evidence
silver_diag = con.execute(f"""
    WITH dim AS (
        SELECT
            symbol,
            ipo_date,
            GREATEST(COALESCE(ipo_date, DATE '{FLOOR_DATE}'), DATE '{FLOOR_DATE}') AS expected_start_date
        FROM {silver('dim_company/*.parquet')}
        WHERE symbol IN {symbols_sql}
    ),
    prices AS (
        SELECT
            symbol,
            MIN(trade_date) AS observed_first_trade_date,
            MAX(trade_date) AS observed_last_trade_date,
            COUNT(*) AS observed_rows
        FROM {silver('fact_daily_prices/**/*.parquet')}
        WHERE symbol IN {symbols_sql}
        GROUP BY symbol
    )
    SELECT
        d.symbol,
        d.ipo_date,
        d.expected_start_date,
        p.observed_first_trade_date,
        p.observed_last_trade_date,
        COALESCE(p.observed_rows, 0) AS observed_rows,
        CASE
            WHEN p.observed_first_trade_date IS NULL THEN NULL
            ELSE datediff('day', d.expected_start_date, p.observed_first_trade_date)
        END AS delta_days
    FROM dim d
    LEFT JOIN prices p USING (symbol)
    ORDER BY d.symbol
""").df()

silver_diag

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,symbol,ipo_date,expected_start_date,observed_first_trade_date,observed_last_trade_date,observed_rows,delta_days
0,CMG,2006-01-26,2006-01-26,2026-01-14,2026-06-25,112,7293
1,MCD,1970-01-02,1999-01-01,2026-01-14,2026-06-25,112,9875
2,SBUX,1992-06-26,1999-01-01,2026-01-14,2026-06-25,112,9875


In [4]:
# Bronze metadata evidence
date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')

def bronze_metadata_for_symbol(symbol: str) -> dict:
    prefix = f'bronze/daily_prices/{symbol}/'
    keys = r2_client.list_keys(prefix)

    pull_dates = []
    for key in keys:
        m = date_pattern.search(key)
        if m:
            pull_dates.append(m.group(0))

    pull_dates = sorted(set(pull_dates))
    earliest_pull = pull_dates[0] if pull_dates else None
    latest_pull = pull_dates[-1] if pull_dates else None

    return {
        'symbol': symbol,
        'bronze_key_count': len(keys),
        'bronze_pull_date_count': len(pull_dates),
        'bronze_earliest_pull_date': earliest_pull,
        'bronze_latest_pull_date': latest_pull,
        'bronze_sample_key': keys[-1] if keys else None
    }

bronze_rows = [bronze_metadata_for_symbol(s) for s in SYMBOLS]
bronze_diag = pd.DataFrame(bronze_rows).sort_values('symbol').reset_index(drop=True)
bronze_diag

c:\Users\Dhrumil\alphavantage-equity-pipeline\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '6cc593a1c930a746bde3b2e7900902f9.r2.cloudflarestorage.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Dhrumil\alphavantage-equity-pipeline\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '6cc593a1c930a746bde3b2e7900902f9.r2.cloudflarestorage.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Dhrumil\alphavantage-equity-pipeline\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '6cc593a1c930a746bde3b2e7900902f9.r2.cloudflarestorage.com'. 

,symbol,bronze_key_count,bronze_pull_date_count,bronze_earliest_pull_date,bronze_latest_pull_date,bronze_sample_key
0,CMG,15,15,2026-06-05,2026-06-25,bronze/daily_prices/CMG/2026-06-25.json
1,MCD,15,15,2026-06-05,2026-06-25,bronze/daily_prices/MCD/2026-06-25.json
2,SBUX,15,15,2026-06-05,2026-06-25,bronze/daily_prices/SBUX/2026-06-25.json


In [5]:
# Merge and classify likely root cause
diag = silver_diag.merge(bronze_diag, on='symbol', how='left')

def classify(row) -> str:
    if row['observed_rows'] == 0:
        return 'missing_prices_in_silver'
    if pd.notna(row['delta_days']) and row['delta_days'] > 0:
        if row.get('bronze_pull_date_count', 0) <= 1:
            return 'likely_compact_window_or_no_historical_backfill'
        return 'likely_latest_file_only_transform_scope'
    if pd.notna(row['delta_days']) and row['delta_days'] < 0:
        return 'starts_before_expected_check_floor_or_ipo_rule'
    return 'coverage_ok'

diag['diagnostic_classification'] = diag.apply(classify, axis=1)
diag

,symbol,ipo_date,expected_start_date,observed_first_trade_date,observed_last_trade_date,observed_rows,delta_days,bronze_key_count,bronze_pull_date_count,bronze_earliest_pull_date,bronze_latest_pull_date,bronze_sample_key,diagnostic_classification
0,CMG,2006-01-26,2006-01-26,2026-01-14,2026-06-25,112,7293,15,15,2026-06-05,2026-06-25,bronze/daily_prices/CMG/2026-06-25.json,likely_latest_file_only_transform_scope
1,MCD,1970-01-02,1999-01-01,2026-01-14,2026-06-25,112,9875,15,15,2026-06-05,2026-06-25,bronze/daily_prices/MCD/2026-06-25.json,likely_latest_file_only_transform_scope
2,SBUX,1992-06-26,1999-01-01,2026-01-14,2026-06-25,112,9875,15,15,2026-06-05,2026-06-25,bronze/daily_prices/SBUX/2026-06-25.json,likely_latest_file_only_transform_scope


In [6]:
# Remediation checklist for selected symbols
remediation = pd.DataFrame({
    'step': [
        'Targeted full ingestion for affected symbols',
        'Re-run transform_daily_prices',
        'Optionally rebuild gold price-derived tables',
        'Rerun price coverage audit notebook'
    ],
    'command_or_action': [
        'Run ingestion/ingest_daily_prices.py with --mode full for affected symbols (or temporary universe file)',
        'python -m transform.transform_daily_prices',
        'python -m transform_gold.build_prices_enriched and python -m transform_gold.build_dim_company_enriched',
        'Recompute df_audit and confirm delta_days moves toward 0'
    ]
})

remediation

,step,command_or_action
0,Targeted full ingestion for affected symbols,Run ingestion/ingest_daily_prices.py with --mo...
1,Re-run transform_daily_prices,python -m transform.transform_daily_prices
2,Optionally rebuild gold price-derived tables,python -m transform_gold.build_prices_enriched...
3,Rerun price coverage audit notebook,Recompute df_audit and confirm delta_days move...


In [ ]:
# Optional: save diagnostics output
out_dir = PROJECT_ROOT / 'notebooks' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

diag.to_csv(out_dir / 'price_coverage_diagnostics.csv', index=False)
print('Wrote diagnostics CSV to', out_dir / 'price_coverage_diagnostics.csv')